# V2G 24-Hour Command Logic with Open-Meteo and Revised Snapshot Rules

This notebook updates the snapshot-based logic in three ways:

1. **Training** uses only the time range where **price data and weather data overlap**.
2. **Inference** uses **Open-Meteo hourly forecast** as live weather input for the next 24 hours.
3. **Decision logic** uses a single fixed 24-hour snapshot:
   - request example: `2026-04-16 17:30:20`
   - snapshot start: `2026-04-16 17:00:00`
   - output horizon: `2026-04-16 17:00:00` → `2026-04-17 16:00:00`

The model runs **once** for that snapshot and returns one JSON command per hour:
- `-1` = discharge
- `0` = do nothing
- `1` = charge

## Revised command rules

- Determine **discharge hours** only on the request day evening, using predicted prices above that evening's average.
- The **charge window starts immediately after the last discharge hour**.
- Charging continues until **07:00**, so only hours `< 7` are eligible on the next day.
- We do **not** use a fixed charge-start hour like 22:00.
- We do **not** use the next-day average price anymore.

In [1]:

import json
import os
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sklearn.ensemble import RandomForestRegressor


## 1. Configuration

In [2]:
DATA_PATH = Path("../data")

PRICE_FILE = "ember-energy.org-electricity-price-hungary-20260308.csv"
WEATHER_FILE = "Budapest_01.02.2005-31.09.2025.csv"

PLUGIN_TIME = 18          # earliest discharge hour
PLUGOUT_TIME = 7          # end of charge window in the morning
CHARGE_START_HOUR = 23    # charging starts at 22:00
SPREAD_THRESHOLD = 0.01   # EUR/kWh

SUPPORTED_START_YEAR = 2021
SUPPORTED_END_YEAR = 2030

BUDAPEST_LAT = 47.4979
BUDAPEST_LON = 19.0402
BUDAPEST_TIMEZONE = "Europe/Budapest"

# Open-Meteo does not require an API key for this notebook.

## 2. Load and prepare hourly price data

In [3]:

price_df = pd.read_csv(DATA_PATH / PRICE_FILE)
price_df.head()


,Country,ISO3 Code,Datetime (UTC),Datetime (Local),Price (EUR/MWhe)
0,Hungary,HUN,2015-01-01 00:00:00,2015-01-01 01:00:00,44.16
1,Hungary,HUN,2015-01-01 01:00:00,2015-01-01 02:00:00,39.17
2,Hungary,HUN,2015-01-01 02:00:00,2015-01-01 03:00:00,26.93
3,Hungary,HUN,2015-01-01 03:00:00,2015-01-01 04:00:00,20.94
4,Hungary,HUN,2015-01-01 04:00:00,2015-01-01 05:00:00,18.52


In [4]:

price_df = price_df.copy()

# Support both original and already-cleaned versions of the price file
if "Datetime (Local)" in price_df.columns:
    price_df["Datetime"] = pd.to_datetime(price_df["Datetime (Local)"])
elif "Datetime" in price_df.columns:
    price_df["Datetime"] = pd.to_datetime(price_df["Datetime"])
else:
    raise ValueError("No datetime column found in the price dataset.")

if "Price (EUR/MWhe)" in price_df.columns:
    price_df["Price"] = price_df["Price (EUR/MWhe)"] / 1000.0
elif "Price" in price_df.columns:
    price_df["Price"] = pd.to_numeric(price_df["Price"], errors="coerce")
else:
    raise ValueError("No price column found in the price dataset.")

price_df = price_df[["Datetime", "Price"]].copy()
price_df["Datetime"] = pd.to_datetime(price_df["Datetime"])
price_df["Price"] = pd.to_numeric(price_df["Price"], errors="coerce")

price_df = (
    price_df.groupby("Datetime", as_index=False)["Price"]
    .mean()
    .sort_values("Datetime")
)

price_ts_df = price_df.set_index("Datetime").asfreq("h")
price_ts_df["Price"] = price_ts_df["Price"].interpolate(method="time").bfill().ffill()

price_ts_df.head()


,Price
Datetime,
2015-01-01 01:00:00,0.04416
2015-01-01 02:00:00,0.03917
2015-01-01 03:00:00,0.02693
2015-01-01 04:00:00,0.02094
2015-01-01 05:00:00,0.01852


## 3. Load and clean historical weather data

In [5]:

weather_df = pd.read_csv(DATA_PATH / WEATHER_FILE, sep=";")
weather_df.head()


,time,temp,p_station,p_sea,humidity,wind_dir,wind_speed,wind_gust_10min,wind_gust,clouds,conditions,visibility,dew,precipitation
0,30.09.2025 23:00,8.1,755.3,768.2,80.0,Wind blowing from the north,1.0,2.0,2.0,no clouds,,20,4.9,No precipitation
1,30.09.2025 22:00,9.0,755.3,768.1,76.0,Wind blowing from the north-northeast,1.0,NaN,3.0,no clouds,,20,5.0,No precipitation
2,30.09.2025 21:00,9.8,755.1,767.9,73.0,Wind blowing from the north-east,1.0,2.0,NaN,"10% or less, but not 0",,20,5.2,No precipitation
3,30.09.2025 20:00,11.9,755.0,767.7,63.0,Wind blowing from the north-east,1.0,2.0,2.0,100%.,,20,5.1,No precipitation
4,30.09.2025 19:00,11.7,754.7,767.4,64.0,Wind blowing from the west-southwest,1.0,1.0,2.0,100%.,,20,5.1,No precipitation


In [6]:

weather_df = weather_df[["time", "temp", "clouds", "visibility", "precipitation"]].copy()
weather_df["Datetime"] = pd.to_datetime(weather_df["time"], dayfirst=True)

clouds_mapping = {
    "no clouds": 0,
    "10%  or less, but not 0": 0.05,
    "20–30%.": 0.25,
    "40%.": 0.4,
    "50%.": 0.5,
    "60%.": 0.6,
    "70 – 80%.": 0.75,
    "90  or more, but not 100%": 0.95,
    "100%.": 1,
    "Sky obscured by fog and/or other meteorological phenomena.": 0,
}

weather_df["temp"] = pd.to_numeric(weather_df["temp"], errors="coerce")
weather_df["clouds"] = weather_df["clouds"].map(clouds_mapping).fillna(0)

weather_df["visibility"] = (
    weather_df["visibility"]
    .astype(str)
    .str.strip()
    .replace({"less than 0.1": "0", "": np.nan})
)
weather_df["visibility"] = pd.to_numeric(weather_df["visibility"], errors="coerce")

weather_df["precipitation"] = (
    weather_df["precipitation"]
    .astype(str)
    .str.strip()
    .replace({"No precipitation": "0", "": np.nan})
)
weather_df["precipitation"] = pd.to_numeric(weather_df["precipitation"], errors="coerce")

weather_features_df = weather_df[[
    "Datetime",
    "temp",
    "clouds",
    "visibility",
    "precipitation",
]].copy()

weather_features_df = (
    weather_features_df
    .sort_values("Datetime")
    .drop_duplicates("Datetime")
)

weather_cols = ["temp", "clouds", "visibility", "precipitation"]
weather_features_df[weather_cols] = weather_features_df[weather_cols].apply(pd.to_numeric, errors="coerce")

weather_features_df.head()


,Datetime,temp,clouds,visibility,precipitation
128238,2005-02-01 01:00:00,2.1,1.00,17.0,0.1
128237,2005-02-01 04:00:00,1.9,1.00,15.0,NaN
128236,2005-02-01 07:00:00,1.9,0.75,25.0,0.1
128235,2005-02-01 10:00:00,1.7,0.75,40.0,NaN
128234,2005-02-01 13:00:00,3.6,0.60,15.0,NaN


## 4. Train the price model on the overlap only

In [7]:

def add_time_features(df, datetime_col="Datetime"):
    df = df.copy()
    dt = pd.to_datetime(df[datetime_col])

    iso = dt.dt.isocalendar()

    df["hour"] = dt.dt.hour
    df["weekday"] = dt.dt.weekday
    df["month"] = dt.dt.month
    df["year"] = dt.dt.year
    df["iso_week"] = iso.week.astype(int)
    df["day_of_year"] = dt.dt.dayofyear
    df["is_weekend"] = (dt.dt.weekday >= 5).astype(int)

    return df


feature_columns = [
    "hour",
    "weekday",
    "month",
    "year",
    "iso_week",
    "day_of_year",
    "is_weekend",
    "temp",
    "clouds",
    "visibility",
    "precipitation",
]

training_start = max(price_ts_df.index.min(), weather_features_df["Datetime"].min())
training_end = min(price_ts_df.index.max(), weather_features_df["Datetime"].max())

print("Training overlap start:", training_start)
print("Training overlap end:  ", training_end)

price_overlap_df = (
    price_ts_df.loc[training_start:training_end]
    .reset_index()
    .rename(columns={"index": "Datetime"})
)

weather_overlap_df = weather_features_df[
    weather_features_df["Datetime"].between(training_start, training_end)
].copy()

price_model_df = price_overlap_df.merge(
    weather_overlap_df,
    on="Datetime",
    how="inner",
)

price_model_df = add_time_features(price_model_df, "Datetime")
price_model_df = price_model_df.sort_values("Datetime")

price_model_df[weather_cols] = price_model_df[weather_cols].ffill().bfill()
for col in weather_cols:
    price_model_df[col] = price_model_df[col].fillna(price_model_df[col].median())

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
)

model.fit(price_model_df[feature_columns], price_model_df["Price"])

price_model_df[["Datetime", "Price"] + weather_cols].head()


Training overlap start: 2015-01-01 01:00:00
Training overlap end:   2025-09-30 23:00:00


,Datetime,Price,temp,clouds,visibility,precipitation
0,2015-01-01 01:00:00,0.04416,-9.0,0.95,4.0,2.0
1,2015-01-01 02:00:00,0.03917,-8.4,0.95,4.0,2.0
2,2015-01-01 03:00:00,0.02693,-7.5,0.95,5.0,2.0
3,2015-01-01 04:00:00,0.02094,-7.0,1.00,5.0,2.0
4,2015-01-01 05:00:00,0.01852,-6.8,1.00,6.0,2.0


## 5. Open-Meteo hourly forecast helpers

In [8]:
def get_open_meteo_hourly_weather_for_request(
    request_timestamp,
    latitude=BUDAPEST_LAT,
    longitude=BUDAPEST_LON,
    timezone=BUDAPEST_TIMEZONE,
):
    """
    Fetch hourly forecast weather for the 24-hour snapshot window anchored to the
    floored request hour.

    Example:
        request_timestamp = "2026-04-16 17:30:20"
        window start      = "2026-04-16 17:00:00"
        window end        = "2026-04-17 16:00:00"

    Returns a dataframe with:
        Datetime, temp, clouds, visibility, precipitation
    """
    request_timestamp = pd.Timestamp(request_timestamp)
    start_hour = request_timestamp.floor("h")
    end_hour = start_hour + pd.Timedelta(hours=23)

    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "temperature_2m,cloud_cover,visibility,precipitation",
        "timezone": timezone,
        "start_hour": start_hour.strftime("%Y-%m-%dT%H:%M"),
        "end_hour": end_hour.strftime("%Y-%m-%dT%H:%M"),
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()

    if "hourly" not in data:
        raise ValueError("Open-Meteo response does not contain 'hourly' data.")

    hourly = data["hourly"]

    weather_df = pd.DataFrame({
        "Datetime": pd.to_datetime(hourly["time"]),
        "temp": pd.to_numeric(hourly["temperature_2m"], errors="coerce"),
        "clouds": pd.to_numeric(hourly["cloud_cover"], errors="coerce") / 100.0,
        "visibility": pd.to_numeric(hourly["visibility"], errors="coerce") / 1000.0,
        "precipitation": pd.to_numeric(hourly["precipitation"], errors="coerce"),
    })

    expected_hours = pd.date_range(start=start_hour, periods=24, freq="h")
    weather_df = (
        pd.DataFrame({"Datetime": expected_hours})
        .merge(weather_df, on="Datetime", how="left")
        .sort_values("Datetime")
    )

    weather_cols = ["temp", "clouds", "visibility", "precipitation"]
    weather_df[weather_cols] = weather_df[weather_cols].ffill().bfill()

    if weather_df[weather_cols].isna().any().any():
        raise ValueError("Missing weather values remained after fill.")

    return weather_df

## 6. Predict 24 hourly prices from the weather forecast

In [9]:
def build_prediction_features_with_weather(
    request_timestamp,
    historical_training_df_for_fallback,
    latitude=BUDAPEST_LAT,
    longitude=BUDAPEST_LON,
    timezone=BUDAPEST_TIMEZONE,
):
    """
    Build the 24-hour feature frame for prediction using:
    - one fixed snapshot window from request_timestamp.floor("h")
    - Open-Meteo hourly forecast for that same 24-hour window
    """
    request_timestamp = pd.Timestamp(request_timestamp)
    start_hour = request_timestamp.floor("h")
    schedule_hours = pd.date_range(start=start_hour, periods=24, freq="h")

    future_df = pd.DataFrame({"Datetime": schedule_hours})
    future_df = add_time_features(future_df, "Datetime")

    weather_forecast_df = get_open_meteo_hourly_weather_for_request(
        request_timestamp=request_timestamp,
        latitude=latitude,
        longitude=longitude,
        timezone=timezone,
    )

    future_df = future_df.merge(weather_forecast_df, on="Datetime", how="left")

    weather_cols = ["temp", "clouds", "visibility", "precipitation"]
    future_df[weather_cols] = future_df[weather_cols].ffill().bfill()

    for col in weather_cols:
        fallback_value = historical_training_df_for_fallback[col].median()
        future_df[col] = future_df[col].fillna(fallback_value)

    return future_df


def build_prediction_frame(
    request_timestamp,
    historical_training_df_for_fallback,
    latitude=BUDAPEST_LAT,
    longitude=BUDAPEST_LON,
    timezone=BUDAPEST_TIMEZONE,
):
    """Build the 24-hour prediction frame for one fixed snapshot window."""
    future_df = build_prediction_features_with_weather(
        request_timestamp=request_timestamp,
        historical_training_df_for_fallback=historical_training_df_for_fallback,
        latitude=latitude,
        longitude=longitude,
        timezone=timezone,
    )

    future_df["Price"] = model.predict(future_df[feature_columns])
    future_df["date"] = future_df["Datetime"].dt.normalize()
    future_df["hour"] = future_df["Datetime"].dt.hour

    return future_df

## 7. Snapshot decision engine

In [16]:
def get_snapshot_metrics(prediction_frame, snapshot_start):
    """
    Build ONE fixed decision snapshot from the already-predicted 24-hour horizon.

    Revised rules:
    - We do not re-run the model for 18:00, 19:00, ...
    - We run it once for the snapshot hour only.
    - Discharge hours are selected only from the request-day evening window.
    - Charge starts immediately after the LAST discharge hour and continues
      through the night until 07:00 (hours < 7).
    - We do not use next-day average price anymore.
    """
    snapshot_start = pd.Timestamp(snapshot_start).floor("h")
    snapshot_end = snapshot_start + pd.Timedelta(hours=23)
    snapshot_day = snapshot_start.normalize()
    next_day = snapshot_day + pd.Timedelta(days=1)

    horizon_df = prediction_frame[
        prediction_frame["Datetime"].between(snapshot_start, snapshot_end)
    ].copy().sort_values("Datetime")

    day_df = horizon_df[horizon_df["date"] == snapshot_day].copy()

    if day_df.empty:
        raise ValueError(f"No predicted price data available for {snapshot_day.date()}.")

    evening_df = day_df[day_df["hour"] >= max(PLUGIN_TIME, snapshot_start.hour)].copy()

    if evening_df.empty:
        discharge_hours_df = horizon_df.iloc[0:0].copy()
        charge_hours_df = horizon_df.iloc[0:0].copy()
        evening_avg_price = np.nan
        avg_discharge_price = np.nan
        avg_charge_price = np.nan
        spread = np.nan
        is_profitable = False
    else:
        evening_avg_price = evening_df["Price"].mean()

        discharge_hours_df = evening_df[
            evening_df["Price"] > evening_avg_price
        ].copy().sort_values("Datetime")

        if discharge_hours_df.empty:
            charge_hours_df = horizon_df.iloc[0:0].copy()
            avg_discharge_price = np.nan
            avg_charge_price = np.nan
            spread = np.nan
            is_profitable = False
        else:
            last_discharge_ts = discharge_hours_df["Datetime"].max()

            charge_hours_df = horizon_df[
                (horizon_df["Datetime"] > last_discharge_ts) &
                (
                    ((horizon_df["date"] == snapshot_day) & (horizon_df["hour"] > CHARGE_START_HOUR)) |
                    ((horizon_df["date"] == next_day) & (horizon_df["hour"] < PLUGOUT_TIME))
                )
            ].copy().sort_values("Datetime")

            avg_discharge_price = discharge_hours_df["Price"].mean()
            avg_charge_price = charge_hours_df["Price"].mean() if not charge_hours_df.empty else np.nan

            spread = (
                avg_discharge_price - avg_charge_price
                if pd.notna(avg_charge_price) else np.nan
            )
            is_profitable = pd.notna(spread) and spread >= SPREAD_THRESHOLD

            if not is_profitable:
                discharge_hours_df = discharge_hours_df.iloc[0:0].copy()
                charge_hours_df = charge_hours_df.iloc[0:0].copy()

    return {
        "snapshot_start": snapshot_start,
        "snapshot_end": snapshot_end,
        "evening_avg_price": evening_avg_price,
        "avg_discharge_price": avg_discharge_price,
        "avg_charge_price": avg_charge_price,
        "spread": spread,
        "is_profitable": bool(is_profitable),
        "discharge_hours": set(discharge_hours_df["Datetime"]),
        "charge_hours": set(charge_hours_df["Datetime"]),
    }


def decide_snapshot_command(ts, snapshot_metrics):
    ts = pd.Timestamp(ts).floor("h")

    if ts < snapshot_metrics["snapshot_start"] or ts > snapshot_metrics["snapshot_end"]:
        raise ValueError("Timestamp is outside the 24-hour snapshot window.")

    if not snapshot_metrics["is_profitable"]:
        return "0"

    if ts in snapshot_metrics["discharge_hours"]:
        return "-1"

    if ts in snapshot_metrics["charge_hours"]:
        return "1"

    return "0"


def build_24h_command_schedule(request_timestamp, historical_training_df_for_fallback, lat=BUDAPEST_LAT, lon=BUDAPEST_LON):
    """
    Single-run snapshot schedule.

    Example:
    request timestamp = 2026-04-16 17:30:20
    first decision hour = 2026-04-16 17:00:00
    last decision hour  = 2026-04-17 16:00:00
    """
    request_timestamp = pd.Timestamp(request_timestamp)
    start_hour = request_timestamp.floor("h")
    year = start_hour.year

    if year < SUPPORTED_START_YEAR or year > SUPPORTED_END_YEAR:
        raise ValueError(
            f"Year {year} is outside the supported model range "
            f"{SUPPORTED_START_YEAR}-{SUPPORTED_END_YEAR}."
        )

    prediction_df = build_prediction_frame(
        request_timestamp=request_timestamp,
        historical_training_df_for_fallback=historical_training_df_for_fallback,
        latitude=lat,
        longitude=lon,
        timezone=BUDAPEST_TIMEZONE,
    )

    snapshot_metrics = get_snapshot_metrics(prediction_df, snapshot_start=start_hour)
    schedule_hours = pd.date_range(start=start_hour, periods=24, freq="h")

    commands = []
    for ts in schedule_hours:
        command = decide_snapshot_command(ts, snapshot_metrics)
        commands.append({ts.strftime("%Y-%m-%d %H:%M:%S"): command})

    return commands, prediction_df, snapshot_metrics


def build_24h_command_schedule_from_predictions(prediction_df, snapshot_start):
    """
    Convenience helper for already-predicted dataframes.
    Applies the same revised snapshot rules.
    """
    prediction_df = prediction_df.copy()
    if "Price" not in prediction_df.columns and "PredictedPrice" in prediction_df.columns:
        prediction_df["Price"] = prediction_df["PredictedPrice"]

    prediction_df["Datetime"] = pd.to_datetime(prediction_df["Datetime"])
    prediction_df["date"] = prediction_df["Datetime"].dt.normalize()
    prediction_df["hour"] = prediction_df["Datetime"].dt.hour

    snapshot_metrics = get_snapshot_metrics(prediction_df, snapshot_start=snapshot_start)

    schedule_hours = pd.date_range(start=pd.Timestamp(snapshot_start).floor("h"), periods=24, freq="h")
    commands = []
    for ts in schedule_hours:
        command = decide_snapshot_command(ts, snapshot_metrics)
        commands.append({ts.strftime("%Y-%m-%d %H:%M:%S"): command})

    return commands, snapshot_metrics

## 8. Simulate a request

In [11]:
# For live use, you can switch this to: request_ts = pd.Timestamp(datetime.now())
request_ts = pd.Timestamp("2026-04-16 17:30:20")

prediction_df = build_prediction_frame(
    request_timestamp=request_ts,
    historical_training_df_for_fallback=price_model_df,
)

prediction_df[[
    "Datetime",
    "temp",
    "clouds",
    "visibility",
    "precipitation",
    "Price",
]].head(24)

,Datetime,temp,clouds,visibility,precipitation,Price
0,2026-04-16 17:00:00,21.2,0.93,28.18,0.0,0.092098
1,2026-04-16 18:00:00,21.0,0.95,50.00,0.0,0.120594
2,2026-04-16 19:00:00,20.3,0.94,27.72,0.0,0.147626
3,2026-04-16 20:00:00,19.2,0.88,26.88,0.0,0.155733
4,2026-04-16 21:00:00,18.3,0.47,50.00,0.0,0.128619
5,2026-04-16 22:00:00,17.5,0.07,23.98,0.0,0.109359
6,2026-04-16 23:00:00,16.6,0.00,22.02,0.0,0.097616
7,2026-04-17 00:00:00,15.3,0.00,50.00,0.0,0.092796
8,2026-04-17 01:00:00,14.8,0.00,50.00,0.0,0.092033
9,2026-04-17 02:00:00,14.5,0.00,50.00,0.0,0.091484


In [17]:
request_ts = pd.Timestamp("2026-04-16 17:30:20")

command_schedule, prediction_df, snapshot_metrics = build_24h_command_schedule(
    request_timestamp=request_ts,
    historical_training_df_for_fallback=price_model_df,
)

command_schedule

[{'2026-04-16 17:00:00': '0'},
 {'2026-04-16 18:00:00': '0'},
 {'2026-04-16 19:00:00': '-1'},
 {'2026-04-16 20:00:00': '-1'},
 {'2026-04-16 21:00:00': '-1'},
 {'2026-04-16 22:00:00': '0'},
 {'2026-04-16 23:00:00': '1'},
 {'2026-04-17 00:00:00': '1'},
 {'2026-04-17 01:00:00': '1'},
 {'2026-04-17 02:00:00': '1'},
 {'2026-04-17 03:00:00': '1'},
 {'2026-04-17 04:00:00': '1'},
 {'2026-04-17 05:00:00': '1'},
 {'2026-04-17 06:00:00': '1'},
 {'2026-04-17 07:00:00': '0'},
 {'2026-04-17 08:00:00': '0'},
 {'2026-04-17 09:00:00': '0'},
 {'2026-04-17 10:00:00': '0'},
 {'2026-04-17 11:00:00': '0'},
 {'2026-04-17 12:00:00': '0'},
 {'2026-04-17 13:00:00': '0'},
 {'2026-04-17 14:00:00': '0'},
 {'2026-04-17 15:00:00': '0'},
 {'2026-04-17 16:00:00': '0'}]

## 9. JSON response

In [13]:
json_response = json.dumps(command_schedule, ensure_ascii=False, indent=2)
print(json_response)

[
  {
    "2026-04-16 17:00:00": "0"
  },
  {
    "2026-04-16 18:00:00": "0"
  },
  {
    "2026-04-16 19:00:00": "-1"
  },
  {
    "2026-04-16 20:00:00": "-1"
  },
  {
    "2026-04-16 21:00:00": "-1"
  },
  {
    "2026-04-16 22:00:00": "1"
  },
  {
    "2026-04-16 23:00:00": "1"
  },
  {
    "2026-04-17 00:00:00": "1"
  },
  {
    "2026-04-17 01:00:00": "1"
  },
  {
    "2026-04-17 02:00:00": "1"
  },
  {
    "2026-04-17 03:00:00": "1"
  },
  {
    "2026-04-17 04:00:00": "1"
  },
  {
    "2026-04-17 05:00:00": "1"
  },
  {
    "2026-04-17 06:00:00": "1"
  },
  {
    "2026-04-17 07:00:00": "0"
  },
  {
    "2026-04-17 08:00:00": "0"
  },
  {
    "2026-04-17 09:00:00": "0"
  },
  {
    "2026-04-17 10:00:00": "0"
  },
  {
    "2026-04-17 11:00:00": "0"
  },
  {
    "2026-04-17 12:00:00": "0"
  },
  {
    "2026-04-17 13:00:00": "0"
  },
  {
    "2026-04-17 14:00:00": "0"
  },
  {
    "2026-04-17 15:00:00": "0"
  },
  {
    "2026-04-17 16:00:00": "0"
  }
]


## 10. Optional app-style wrapper

In [ ]:
def get_app_response(request_timestamp=None, lat=BUDAPEST_LAT, lon=BUDAPEST_LON):
    if request_timestamp is None:
        request_timestamp = datetime.now()

    schedule, prediction_df, snapshot_metrics = build_24h_command_schedule(
        request_timestamp=request_timestamp,
        historical_training_df_for_fallback=price_model_df,
        lat=lat,
        lon=lon,
    )

    return {
        "request_timestamp": pd.Timestamp(request_timestamp).strftime("%Y-%m-%d %H:%M:%S"),
        "snapshot_start": pd.Timestamp(request_timestamp).floor("h").strftime("%Y-%m-%d %H:%M:%S"),
        "snapshot_end": (pd.Timestamp(request_timestamp).floor("h") + pd.Timedelta(hours=23)).strftime("%Y-%m-%d %H:%M:%S"),
        "model_year": pd.Timestamp(request_timestamp).year,
        "commands_next_24h": schedule,
        "spread": None if pd.isna(snapshot_metrics["spread"]) else float(snapshot_metrics["spread"]),
    }

# Example:
# app_response = get_app_response(request_timestamp=request_ts)
# print(json.dumps(app_response, ensure_ascii=False, indent=2))

## 11. Notes

- Training uses **only the overlap** between the historical price series and the historical weather series.
- Inference uses **hourly forecast weather** from **Open-Meteo**.
- The model is evaluated **once per request snapshot**, not once per hour.
- Revised decision logic:
  - discharge during the evening if predicted price is above that evening's average
  - start charging immediately after the last discharge hour
  - continue charging only until **07:00** (`hour < 7`)
  - do not use the next-day average price